In [0]:
%pip install uszipcode
%pip install h3
%pip install keplergl 

In [0]:
dbutils.library.restartPython()

In [0]:
# from uszipcode import SearchEngine
import sqlite3
import pandas as pd
from pyspark.sql.functions import udf, col
from pyspark.sql.types import IntegerType
import math
from urllib import request
import os

In [0]:
H3_RES = 8

In [0]:
BAD_ZIPCODE_VALUE = 'bad_zipcode'
file_location = "dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/"
file_type = "csv"
target_year = 2016

In [0]:
def get_data_files(yyyy, months):
  data_files = []
  for mm in months:
    mm = str(mm) if mm >= 10 else f"0{mm}"
    month_data_files = list(filter(lambda file_name: f"{yyyy}-{mm}" in file_name,
                           [f.path for f in dbutils.fs.ls(file_location)]))
    data_files += month_data_files
  return data_files
  
def load_data(data_files, sample=1.0):
  df = (spark.read.format("csv")
        .option("inferSchema", "true")
        .option("header", "true")
        .option("ignoreLeadingWhiteSpace", "true")
        .option("ignoreTrailingWhiteSpace", "true")
        .option("sep", ",")
        .load(data_files)
      ).sample(False, sample, 123)
  
  # Rename, cast types, and filter columns
  column_allow_list = { 
    "pickup_datetime": ["tpep_pickup_datetime", "timestamp"],
    "tpep_pickup_datetime": ["tpep_pickup_datetime", "timestamp"],
    
    # type conversion
    "dropoff_datetime": ["tpep_dropoff_datetime", "timestamp"],
    "tpep_dropoff_datetime": ["tpep_dropoff_datetime", "timestamp"],
    
    "pickup_zip": ["pickup_zip", "integer"],
    "dropoff_zip": ["dropoff_zip", "integer"],
    "trip_distance": ["trip_distance", "double"],
    "fare_amount": ["fare_amount", "double"],
    "pickup_latitude": ["pickup_latitude", "double"],
    "pickup_longitude": ["pickup_longitude", "double"],
    "dropoff_latitude": ["dropoff_latitude", "double"],
    "dropoff_longitude": ["dropoff_longitude", "double"],
  }
  columns = []
  for orig in df.columns:
    orig_lower = orig.lower()
    if orig_lower in column_allow_list:
      new_name, data_type = column_allow_list[orig_lower]
      columns.append(col(orig).cast(data_type).alias(new_name.lower()))
  
  return df.select(columns)  

In [0]:
# load data and put 

In [0]:
# Generate data file names for the first 2 months of data in 2016
data_files = get_data_files(target_year,months=[1,2])
# Load in a small subsample of data to speed things up for this example
df_taxi_data = load_data(data_files, sample=.001)
display(df_taxi_data)

In [0]:
# df_taxi_data.cache()

In [0]:
# create a temprary view taxi_data
df_taxi_data.createOrReplaceTempView("taxi_data")
# DBTITLE 1,Create a table taxi_data
df_taxi_data_with_h3_index = spark.sql(f"""
SELECT *
    , h3_longlatash3(pickup_longitude, pickup_latitude, {H3_RES}) as pickup_h3_index
    , h3_longlatash3(dropoff_longitude, dropoff_latitude, {H3_RES}) as dropoff_h3_index
FROM taxi_data
""")
display(df_taxi_data_with_h3_index)

In [0]:
df_taxi_data_with_h3_index.createOrReplaceTempView("taxi_data_with_h3_index")
df_taxi_pickup_aggregate = spark.sql("""
    select 
        count(*) as pickup_count
        , sum(fare_amount) as pickup_fare_amount
        , sum(trip_distance) as pickup_trip_distance
        , pickup_h3_index
    from taxi_data_with_h3_index
    group by pickup_h3_index
""")
display(df_taxi_pickup_aggregate)
# df_taxi_data_aggregate

In [0]:
# df_taxi_pickup_aggregate.cache()

In [0]:
from keplergl import KeplerGl
import pandas as pd

print("Kepler.gl imported")

In [0]:
# display(df_taxi_pickup_aggregate
#         .withColumn("h3_index", col("pickup_h3_index"))
#         .withColumn("count", col("pickup_count"))
#         .select("h3_index","count"))

pdf_taxi_pickup = df_taxi_pickup_aggregate\
    .withColumn("h3", col("pickup_h3_index"))\
    .withColumn("fare_amount", col("pickup_fare_amount"))\
    .withColumn("trip_distance", col("pickup_trip_distance"))\
    .withColumn("value", col("pickup_trip_distance"))\
    .withColumn("count", col("pickup_count"))\
    .select("h3","value","count")\
    .toPandas()
display(pdf_taxi_pickup)

In [0]:
# Example input: Spark or pandas
# Case A: starting from Spark DataFrame `df_hex` with columns: h3_index (string), count (numeric)
# df_hex = spark.createDataFrame([("8a2a1072b59ffff", 10), ("8a2a1072b5bffff", 25)], ["h3_index","count"]).cache()

# Convert Spark -> pandas if needed
import h3
try:
    df_h3 = pdf_taxi_pickup
    # convert df_h3's column 'h3' from integer to hex via h3.int_to_str()
    df_h3["h3"] = df_h3["h3"].apply(h3.int_to_str)
except NameError:
    # Fallback: create a small pandas sample
    df_h3 = pd.DataFrame({
        # 617733123820224511
        # 617733122632712191
        "h3_index": ["617733151012945919", "617733123820224511", "617733122632712191"],
        "count": [10, 25, 5]
    })

df_h3.head()

In [0]:
# Build Kepler map using H3 aggregation
# Kepler supports H3 hex bin layer directly via column name

# Minimal config: heatmap by count using H3 column
center_lat, center_lon = 40.73, -73.94  # NYC-ish
m = KeplerGl(height=600)

# Add the aggregated H3 dataset
m.add_data(data=df_h3, name="h3_hexes")

# Kepler H3 layer config:
# - 'type': 'h3' (H3 Hexagon layer)
# - columns.hex_id must point to the column containing the H3 index (here 'h3')
h3_config = {
    "version": "v1",
    "config": {
        "visState": {
            "layers": [
                {
                    "id": "h3-heat",
                    "type": "h3",
                    "config": {
                        "dataId": "h3_hexes",
                        "label": "H3 Heat (res=8)",
                        "columns": {"hex_id": "h3"},
                        "isVisible": True,
                        "visConfig": {
                            "opacity": 0.9,
                            "coverage": 1.0,
                            "enable3d": False,  # set True to extrude by size if desired
                            "strokeColor": None,
                            "colorRange": {
                                "name": "ColorBrewer OrRd-6",
                                "type": "sequential",
                                "category": "ColorBrewer",
                                "colors": ["#fee8c8", "#fdd49e", "#fdbb84", "#fc8d59", "#e34a33", "#b30000"]
                            },
                            "sizeRange": [0, 500],  # used if enable3d=True
                        },
                    },
                    "visualChannels": {
                        # Color the hexes by the aggregated 'value'
                        "colorField": {"name": "value", "type": "real"},
                        "colorScale": "quantile",
                        # If you enable 3D, set 'sizeField' to value or count
                        # "sizeField": {"name": "value", "type": "real"},
                        # "sizeScale": "linear",
                    },
                }
            ],
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "h3_hexes": [
                            {"name": "h3", "format": None},
                            {"name": "value", "format": ".2f"},
                            {"name": "count", "format": None},
                        ]
                    },
                    "enabled": True
                }
            },
        },
        "mapState": {
            "latitude": center_lat,
            "longitude": center_lon,
            "zoom": 9,
            "bearing": 0,
            "pitch": 0,
            "dragRotate": False,
        },
        # "mapStyle": {"styleType": "dark"}
    }
}

m.config = h3_config

# ---- 4) Display inline (Jupyter renders the last expression) ----
m

In [0]:
df_h3